# Build the client-level table

Merges client demographics with the experiment assignment (Test/Control) and aggregates visit-level KPIs up to one row per client.

**Per-client KPIs computed**
- `session_per_client` — number of visits the client made
- `max_step_client` — furthest step ever reached across all their visits
- `n_backward_steps` — total backward navigations across all visits
- `completion_rate` — fraction of the client's visits that completed the funnel
- `avg_steps_per_session` — average events per visit
- `error_rate` — error rate ?
- `avg_time_to_completion` — average duration of completed visits (NaN if client never completed)

## Load source tables

In [1]:
import pandas as pd

df1 = pd.read_csv('df_demo_clean.csv')
df2 = pd.read_csv('df_final_experiment_clients.csv')
df_events = pd.read_csv('events_kpi_table.csv')
df_visits = pd.read_csv('visits_kpi_table.csv')

## Merge demographics with experiment assignment

Inner join keeps only clients who appear in both tables (i.e., demo info exists *and* they were assigned to a group).

In [2]:
df_clients_roster = df1.merge(df2, on='client_id', how='inner')

In [3]:
# Drop clients without an assigned Variation. They don't belong to either Test or Control
df_clients_roster = df_clients_roster.dropna(subset=['Variation'])

In [4]:
# 'X' is an unclear/edge gender code. Let's fold into 'U' (unknown) for clean grouping later
df_clients_roster['gendr'] = df_clients_roster['gendr'].replace({'X': 'U'})

## Aggregate visit-level KPIs to client level

In [14]:
summary = df_visits.groupby("client_id").agg(
    session_per_client =("visit_id", "count"),
    max_step_client = ("max_step_reached", "max"),
    completion_rate = ("reached_confirm", 'max'),
    avg_steps_client = ("n_steps_session", "mean"),
    error_rate=("went_backwards", "max")).reset_index()

summary


,client_id,session_per_client,max_step_client,completion_rate,avg_steps_client,error_rate
0,169,1,4,True,5.000000,False
1,336,1,0,False,2.000000,False
2,546,1,4,True,5.000000,False
3,555,1,4,True,5.000000,False
4,647,1,4,True,5.000000,False
...,...,...,...,...,...,...
120152,9999729,3,4,True,3.666667,True
120153,9999768,1,4,True,12.000000,True
120154,9999832,1,1,False,2.000000,False
120155,9999839,1,4,True,6.000000,False


### Average time to completion

Computed separately because it needs to filter to *completed* visits only. Averaging over all visits (including abandoned ones) would be meaningless. Clients who never completed will end up with `NaN` here, which is correct.

In [15]:
time_completion = df_visits[df_visits["reached_confirm"]].groupby("client_id").agg(
    avg_time_to_completion = ("visit_duration_sec", "mean")).reset_index()

time_completion

,client_id,avg_time_to_completion
0,169,213.0
1,546,133.0
2,555,158.0
3,647,377.0
4,722,599.0
...,...,...
81140,9999451,168.0
81141,9999729,75.0
81142,9999768,486.0
81143,9999839,248.0


## Merge KPIs into the client roster

`how='left'` keeps every client in the roster, even if they have no visits or no completions.

In [16]:
summary_table_client = df_clients_roster.merge(summary, on='client_id', how='left').merge(time_completion, on='client_id', how='left')

In [17]:
summary_table_client

,client_id,clnt_tenure_mnth,clnt_age,gendr,num_accts,bal,calls_6_mnth,logons_6_mnth,Variation,session_per_client,max_step_client,completion_rate,avg_steps_client,error_rate,avg_time_to_completion
0,836976,73.0,60.5,U,2.0,45105.30,6.0,9.0,Test,2,4,True,5.500000,False,1785.000000
1,2304905,94.0,58.0,U,2.0,110860.30,6.0,9.0,Control,1,4,True,6.000000,False,295.000000
2,1439522,64.0,32.0,U,2.0,52467.79,6.0,9.0,Test,2,3,False,2.500000,False,NaN
3,1562045,198.0,49.0,M,2.0,67454.65,3.0,6.0,Test,1,0,False,1.000000,False,NaN
4,5126305,145.0,33.0,F,2.0,103671.75,0.0,3.0,Control,1,0,False,1.000000,False,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50482,1780858,262.0,68.5,M,3.0,372100.59,6.0,9.0,Test,2,4,True,6.000000,False,632.000000
50483,6967120,260.0,68.5,M,3.0,4279873.38,6.0,9.0,Control,1,4,True,5.000000,False,199.000000
50484,5826160,249.0,56.5,F,2.0,44837.16,2.0,5.0,Test,3,4,True,3.333333,False,221.666667
50485,8739285,229.0,69.5,F,2.0,44994.24,1.0,4.0,Test,1,4,True,5.000000,False,692.000000


## Save

In [18]:
summary_table_client.to_csv("client_kpi_table.csv", index=False)